In [ ]:
!uv run python -c "import sys; print(sys.executable)"

/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/bin/python3


In [ ]:
# To import methods from src folder we need to write below code
# By default this file thinks that notebook folder is the root folder we need to point to server as root folder

import sys
import os 
from pathlib import Path 

In [20]:
# Add the project root to python path so we can import src
# Get current working diretory
cwd = Path().resolve()
cwd

PosixPath('/home/sam/my-study/agentic-ai/multimodal-rag-application/server/notebooks')

In [21]:
cwd.name

'notebooks'

In [22]:
cwd.parent

PosixPath('/home/sam/my-study/agentic-ai/multimodal-rag-application/server')

In [23]:
# if we are in notebook directory go one level up
if cwd.name == "notebooks":
    project_root = cwd.parent 
else:
    project_root = cwd 

In [24]:
sys.path

['/home/sam/my-study/agentic-ai/multimodal-rag-application/server',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python311.zip',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/lib/python3.11/site-packages']

In [25]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [26]:
sys.path # sys.path is the list of directories Python searches when you do an import

['/home/sam/my-study/agentic-ai/multimodal-rag-application/server',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python311.zip',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11',
 '/home/sam/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/lib/python3.11/site-packages']

In [33]:
from langchain.agents import create_agent
from langchain.tools import tool
from src.rag.retrieval.index import retrieve_context
from src.rag.retrieval.utils import prepare_prompt_and_invoke_llm
from langgraph.graph import MessagesState
from typing import Any, Dict, List
from langgraph.types import Command 
from langchain_core.tools.base import InjectedToolCallId
from langchain_core.messages import ToolMessage

LangGraph reads Annotated to find reducers

citations: Annotated[List[Dict[str, Any]], lambda x, y: x + y]


where List[Dict[str, Any]] - each item is a dict where keys are strings, values can be anything




where the lambda is a reducer function



lamdba takkes current value x in state and add with y new value rturned by node -appends citations

You canl also write this basic typedict as well

`from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class CustomAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    citations: Annotated[List[Dict[str, Any]], lambda x, y: x + y]`



In [ ]:
# =============================================================================
# STATE DEFINITION
# =============================================================================


from typing import Annotated

# Extend Messagesstate to store citations
class CustomAgentState(MessagesState):
    """Extended Agent state with citations tracking"""
    # citations will accumulate across tool calls
    citations: Annotated[List[Dict[str, Any]], lambda x,y : x + y] = []
    guradrails_passed: bool = True

In [ ]:
# =============================================================================
# PROMPTS
# =============================================================================

BASE_SYSTEM_PROMPT = """You are a helpful AI assistant with access to a RAG (Retrieval-Augmented Generation) tool that searches project-specific documents.

For every user question:

1. Do not assume any question is purely conceptual or general.  
2. Use the `rag_search` tool immediately with a clear and relevant query derived from the user's question. 
3. Use the chat history to understand the context and references in the current question. 
4. Carefully review the retrieved documents and base your entire answer on the RAG results.  
5. If the retrieved information fully answers the user's question, respond clearly and completely using that information.  
6. If the retrieved information is insufficient or incomplete, explicitly state that and provide helpful suggestions or guidance based on what you found.  
7. Always present answers in a clear, well-structured, and conversational manner.

**Make sure to call the rag_search tool correctly**
**Never answer without first querying the RAG tool. This ensures every response is grounded in project-specific context and documentation.**
"""

Create a rag tool bound to project_id under a function - this is called factory function pattern

Problem this solves

- The rag_search tool needs project_id to know which project documents to search. But langchain @tool only reeceives arguments that LLMdeciudes to pass and you never want the LLM choosing the project id



we are following a 2 call LLM approach

RAG tool has one job retieve and sythesize context
The agent has one job  - orehestrate

Agent LLM          → decides when/what to search, formats final response
RAG LLM (inner)    → grounds answer strictly to retrieved context

Images work cleanly — prepare_prompt_and_invoke_llm already handles the multimodal message structure correctly.


In [65]:

def create_rag_tool(project_id: str):
    """Create a RAG search tool bound to a specific project.
    This factory function creates a tool that is bound to a specific project_id,
    allowing the agent to search through that project's documents.

    Args:
        project_id: The UUID of the project whose documents should be searchable

    Retruns:
      A LangChain tool configured for RAG search on the specified project
    """

    @tool
    def rag_search_tool(query:str, tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        """
        Search through project documents using RAG(Retrieval Agumented Generation). 
        This tool retrieves relevant context from current project documents based on the query

        Args:
            query: The search query or question to find relevant information

        Returns:
            A formatted string containing the retrieved context and answer based on this documents
        """

        try:

            # Reteive context using the exisiting RAG pipleine
            texts, images, tables, citations = retrieve_context(project_id, query)
            
            # If texts is empty, it almost certainly means the query found nothing — images and tables without text context are not useful alon
            # if no context found
            if not texts:
                return Command(
                    update = {
                        "messages" : [
                            ToolMessage(
                                "No relevant information found in the project documents for this query",
                                tool_call_id =tool_call_id
                            )
                        ]
                    }
                )

            # Prepate the response using the existing LLM preparation function
            response = prepare_prompt_and_invoke_llm(
                    user_query=query,
                    texts= texts,
                    images = images,
                    tables = tables )

            print(f"\n--- RESPONSE BEING SENT TO LLM ---")
            print(response)

            # return texts, images, tables, citations
            return Command(
                update = {
                    "messages" : [
                        ToolMessage(
                            content = response, # agent LLM reads this
                            tool_call_id = tool_call_id
                        )
                    ],
                    "citations" : citations,
                    
                }
            )
            

        except Exception as e:
            return Command(
                update = {
                    "messages": [
                        ToolMessage(
                            f"Error retrieving information: {str(e)}",
                            tool_call_id=tool_call_id
                        )
                    ]
                }
            )

    return rag_search_tool  # ✅ factory returns the tool — outside the tool function   
            
        
        

because rag_search_tool is wrapped with a tool wrapper, the langchain expecs the return to be
- string
- command
- ToolMessage 
for that reason we are not doing return texts, images, tables, citations as LC cannot accept tuple

if the agent wants to see the all retrieved context serialize into tool message
`





In [66]:
project_id = "8d765605-a676-4b96-a7a9-15be6aa38447"
query = {"messages": [{"role": "user", "content": "What are the two types of sleep?"}]}
rag_tool = create_rag_tool(project_id)

# result = rag_tool.invoke({
#     "query": "What are the two types of sleep?",
#     "tool_call_id": "test-001"
# })

result = rag_tool.invoke({
    "args": {"query": "What are the two types of sleep?"},
    "name": "rag_search_tool",
    "type": "tool_call",
    "id": "test-001"
})

Found document IDs:  9

 RAG STRATEGY: MULTI-QUERY-HYBRID


/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryVariations(queries=[...the two sleep stages?']), input_type=QueryVariations])
  return self.__pydantic_serializer__.to_python(


✅ Generated 4 query variations
Queries: ['What are the two categories of sleep?', 'Can you list the two forms of sleep?', 'What are the two kinds of sleep?', 'What are the two sleep stages?']
Generated 5 query variations for hybrid search
📈 Vector search returned: 4 chunks
📈 Keyword search returned: 2 chunks

--- FINAL RANKED RESULTS ---
  rank=1 | score=0.016393 | 0b8ccb3c… | Sleep consists of two main types: non-REM (NREM) sleep and REM (rapid eye movement) sleep. NREM sleep has three stages, progressing from light to deep sleep. Stage 3 NREM, also called slow-wave sleep,…
  rank=2 | score=0.011290 | ec8bc27f… | Experience-dependent plasticity allows the brain to adapt to individual experiences. London taxi drivers, who must memorize complex city routes, show enlarged hippocampi compared to control subjects. …
  rank=3 | score=0.011111 | 92e0916e… | Long-term memory has essentially unlimited capacity and can last a lifetime. It's divided into explicit (declarative) memory and implici

In [67]:
# dig into the update
print("\n--- STATE UPDATE ---")
print(result)


--- STATE UPDATE ---
Command(update={'messages': [ToolMessage(content='The two main types of sleep are non-REM (NREM) sleep and REM (rapid eye movement) sleep.', tool_call_id='test-001')], 'citations': [{'chunk_id': '0b8ccb3c-97c7-42d6-a3fc-b5b78b927391', 'document_id': '8e79109f-6976-488c-b73a-c483567803ea', 'filename': 'neuroscience.txt', 'page': 6}, {'chunk_id': 'ec8bc27f-6bbe-4fe8-99a2-cfa60cff66a2', 'document_id': '8e79109f-6976-488c-b73a-c483567803ea', 'filename': 'neuroscience.txt', 'page': 5}, {'chunk_id': '92e0916e-59da-4a8c-b437-27334bc7e936', 'document_id': '8e79109f-6976-488c-b73a-c483567803ea', 'filename': 'neuroscience.txt', 'page': 4}, {'chunk_id': '44fd0753-1fc7-4ae7-9f75-7cf8c79f6e2a', 'document_id': '8e79109f-6976-488c-b73a-c483567803ea', 'filename': 'neuroscience.txt', 'page': 7}, {'chunk_id': '38ab0030-fdd1-405d-913a-c54dd9be9c9a', 'document_id': '8e79109f-6976-488c-b73a-c483567803ea', 'filename': 'neuroscience.txt', 'page': 2}]})


In [69]:
# Create the agent 
def create_simple_agent(project_id, model:str="gpt-4o"):
    """Create an agent with RAG tool for specific project"""

    # Create tols list with project-specific RAG tool
    tools = [create_rag_tool(project_id)]

    # Define system prompt
    system_prompt = """You are a helpful AI assistant with access to a RAG (Retrieval Agumented Generation)
    
    For every user question:

    1. Do not assume any question is purely conceptual or general.
    2. Use the 'rag_search' tool immediately with a clear and relevant query derived from the user's question.
    3. Carefully review the retrieved documents and base yout entire answer on the RAG results.
    4. if the retrieved information fully answers the user's question, respond clearly and completely using that information.
    5. If the retireved information is insufficient or incomplete, explicitly state that and provide helpful suggestions or guidance based on what you found.
    6. Always present answers in a clear, well strucutred, and conversational manner.
    
    **Never answer without first querying the RAG tool. This ensures every response is grounded in project specific contest and documentation.**
     """

    # create the agent graph
    agent = create_agent(
        model=model,
        tools = tools,
        system_prompt= system_prompt,
        state_schema = CustomAgentState # Without state_schema, LangGraph agents only know about messages
    )

    return agent

In [71]:

project_id = "8d765605-a676-4b96-a7a9-15be6aa38447"
rag_agent = create_simple_agent(project_id=project_id, model="gpt-4o")

In [72]:
input = {"messages": "What are the two types of sleep?"}
result = rag_agent.invoke(input)
result

Found document IDs:  9

 RAG STRATEGY: MULTI-QUERY-HYBRID


/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryVariations(queries=[...kinds of sleep phases']), input_type=QueryVariations])
  return self.__pydantic_serializer__.to_python(


✅ Generated 4 query variations
Queries: ['different categories of sleep', 'varieties of sleep stages', 'types of sleep cycles', 'kinds of sleep phases']
Generated 5 query variations for hybrid search
📈 Vector search returned: 2 chunks
📈 Keyword search returned: 2 chunks

--- FINAL RANKED RESULTS ---
  rank=1 | score=0.016393 | 0b8ccb3c… | Sleep consists of two main types: non-REM (NREM) sleep and REM (rapid eye movement) sleep. NREM sleep has three stages, progressing from light to deep sleep. Stage 3 NREM, also called slow-wave sleep,…
  rank=2 | score=0.011290 | ec8bc27f… | Experience-dependent plasticity allows the brain to adapt to individual experiences. London taxi drivers, who must memorize complex city routes, show enlarged hippocampi compared to control subjects. …
  rank=3 | score=0.004839 | 38ab0030… | === NEUROTRANSMITTERS AND NEUROMODULATORS ===

Neurotransmitters are chemical messengers that enable communication between neurons. There are many different types, each with s

{'messages': [HumanMessage(content='What are the two types of sleep?', additional_kwargs={}, response_metadata={}, id='71b0555e-46f2-4c18-84dd-df802279a97f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 293, 'total_tokens': 312, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_bcd81581f8', 'id': 'chatcmpl-DoR9m06oxpBkpuPqboS4YuXsDouwM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea6b2-44db-72f0-9174-49801fca9f02-0', tool_calls=[{'name': 'rag_search_tool', 'args': {'query': 'two types of sleep'}, 'id': 'call_hbAsNG2hMzeFd0wgRdraMMCa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

In [74]:
result ["messages"][-1].content

'The two main types of sleep are non-REM (NREM) sleep and REM (rapid eye movement) sleep:\n\n1. **Non-REM (NREM) Sleep**:\n   - This type of sleep consists of three stages, progressing from light sleep to deep sleep.\n   - Stage 3 NREM is the deepest and is crucial for physical restoration and memory consolidation.\n\n2. **REM Sleep**:\n   - REM sleep is characterized by rapid eye movements, vivid dreams, and temporary muscle paralysis.\n   - During this stage, brain activity resembles that of wakefulness. It plays a key role in emotional processing, creativity, and the consolidation of procedural memories.'